# 03 Outputs and Summary

## Final Individual Project

**Project topic:** Screen Time and BMI Percentile among High School Students

This notebook organizes the final outputs from the previous notebooks and prepares materials for:

- README
- one-page infographic summary
- presentation video
- final project submission

This notebook does **not** run a new statistical method.  
It summarizes the data cleaning and simple linear regression results.


In [24]:
# Import packages
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)


## 1. Set project paths

This notebook is designed to be placed inside the `notebooks/` folder.


In [25]:
# Project paths
project_dir = Path("..").resolve()

processed_dir = project_dir / "data" / "processed"
figures_dir = project_dir / "outputs" / "figures"
tables_dir = project_dir / "outputs" / "tables"
summary_dir = project_dir / "outputs" / "summary"
report_dir = project_dir / "report"

# Create folders if they do not exist
processed_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
summary_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)


Project directory: D:\作業\統計學實習\final individual project


## 2. Load key outputs from previous notebooks

This notebook uses files created by:

- `01_data_cleaning.ipynb`
- `02_main_regression.ipynb`


In [26]:
# Required files
cleaning_summary_path = tables_dir / "data_cleaning_summary.csv"
descriptive_path = tables_dir / "regression_descriptive_statistics.csv"
key_results_path = tables_dir / "regression_key_results.csv"
compact_summary_path = summary_dir / "compact_result_summary.txt"

required_files = [
    cleaning_summary_path,
    descriptive_path,
    key_results_path,
    compact_summary_path
]

missing_files = [file for file in required_files if not file.exists()]

if missing_files:
    raise FileNotFoundError(
        "Some required output files are missing. "
        "Please run 01_data_cleaning.ipynb and 02_main_regression.ipynb first. "
        f"Missing files: {missing_files}"
    )

cleaning_summary = pd.read_csv(cleaning_summary_path)
descriptive_stats = pd.read_csv(descriptive_path, index_col=0)
key_results = pd.read_csv(key_results_path)

with open(compact_summary_path, "r", encoding="utf-8") as f:
    compact_summary = f.read()

print("Loaded all required files successfully.")


Loaded all required files successfully.


## 3. Review cleaning and regression outputs


In [27]:
cleaning_summary


,Step,Rows
0,Raw dataset,14041
1,Selected variables,14041
2,Complete cases after recoding and removing mis...,12844


In [28]:
descriptive_stats


,count,mean,std,min,25%,50%,75%,max
screen_time_hours,12844.0000,3.7670,2.4868,0.0000,2.0000,3.5000,5.0000,10.0000
bmi_percentile,12844.0000,64.8381,27.5137,0.0000,45.1714,70.2007,89.4428,99.9392


In [29]:
key_results


,Statistic,Value
0,Sample size,12844.0000
1,Intercept,61.6837
2,Slope for screen_time_hours,0.8374
3,t statistic for slope,8.6015
4,p-value for slope,0.0000
5,95% CI lower for slope,0.6466
6,95% CI upper for slope,1.0282
7,R-squared,0.0057
8,Adjusted R-squared,0.0057


## 4. Extract final result values

The next cells extract the key values needed for the README, infographic, and presentation script.


In [30]:
def get_value(statistic_name):
    value = key_results.loc[key_results["Statistic"] == statistic_name, "Value"]
    if value.empty:
        raise KeyError(f"Could not find statistic: {statistic_name}")
    return float(value.iloc[0])

n = int(get_value("Sample size"))
intercept = get_value("Intercept")
slope = get_value("Slope for screen_time_hours")
t_stat = get_value("t statistic for slope")
p_value = get_value("p-value for slope")
ci_lower = get_value("95% CI lower for slope")
ci_upper = get_value("95% CI upper for slope")
r_squared = get_value("R-squared")
adj_r_squared = get_value("Adjusted R-squared")

# p-value display
if p_value < 0.001:
    p_display = "p < 0.001"
else:
    p_display = f"p = {p_value:.4f}"

# Direction and conclusion wording
if slope > 0:
    direction = "positive"
    direction_sentence = "Higher screen time was associated with slightly higher BMI percentile."
elif slope < 0:
    direction = "negative"
    direction_sentence = "Higher screen time was associated with slightly lower BMI percentile."
else:
    direction = "zero"
    direction_sentence = "Screen time did not show an estimated linear change in BMI percentile."

if p_value < 0.05:
    significance_sentence = "The relationship was statistically significant."
else:
    significance_sentence = "The relationship was not statistically significant."

effect_sentence = (
    f"For each additional hour of screen time, the predicted BMI percentile increased by about "
    f"{slope:.2f} percentile points on average."
)

r2_sentence = (
    f"However, screen time explained only about {r_squared * 100:.2f}% of the variation in BMI percentile."
)

print("Extracted values:")
print("n =", n)
print("intercept =", round(intercept, 4))
print("slope =", round(slope, 4))
print("95% CI =", [round(ci_lower, 4), round(ci_upper, 4)])
print("p-value display =", p_display)
print("R-squared =", round(r_squared, 4))


Extracted values:
n = 12844
intercept = 61.6837
slope = 0.8374
95% CI = [0.6466, 1.0282]
p-value display = p < 0.001
R-squared = 0.0057


## 5. Create final key results table

This table is designed for the README and one-page infographic.


In [31]:
final_key_results = pd.DataFrame({
    "Item": [
        "Research question",
        "Method",
        "Sample size",
        "Regression equation",
        "Slope",
        "95% CI for slope",
        "p-value",
        "R-squared",
        "Main interpretation"
    ],
    "Result": [
        "Is there a linear relationship between screen time and BMI percentile among high school students?",
        "Simple Linear Regression",
        f"{n}",
        f"Predicted BMI percentile = {intercept:.4f} + {slope:.4f} × Screen time hours",
        f"{slope:.4f}",
        f"[{ci_lower:.4f}, {ci_upper:.4f}]",
        p_display,
        f"{r_squared:.4f}",
        "Statistically significant but practically weak positive association"
    ]
})

final_key_results


,Item,Result
0,Research question,Is there a linear relationship between screen ...
1,Method,Simple Linear Regression
2,Sample size,12844
3,Regression equation,Predicted BMI percentile = 61.6837 + 0.8374 × ...
4,Slope,0.8374
5,95% CI for slope,"[0.6466, 1.0282]"
6,p-value,p < 0.001
7,R-squared,0.0057
8,Main interpretation,Statistically significant but practically weak...


In [32]:
# Save final key results table
final_key_results.to_csv(tables_dir / "final_key_results_for_report.csv", index=False)

print("Saved:")
print(tables_dir / "final_key_results_for_report.csv")


Saved:
D:\作業\統計學實習\final individual project\outputs\tables\final_key_results_for_report.csv


## Overall Interpretation
Simple linear regression showed a statistically significant positive relationship between screen time and BMI percentile. Each additional hour of screen time was associated with an increase of about 0.84 BMI percentile points on average. However, the R-squared value was only 0.0057, meaning that screen time explained only about 0.57% of the variation in BMI percentile. Therefore, the relationship was statistically significant but practically weak. Since the data are observational, the result should be interpreted as association, not causation.